In [17]:
!pip install transformers==4.44.0 -q

In [18]:
!pip install --upgrade Pillow -q

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 57.8 MB/s eta 0:00:0000:0100:01


In [19]:
!pip install accelerate -q

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 KB 9.3 MB/s eta 0:00:00a 0:00:01


In [20]:
!pip install --upgrade jinja2>=3.1.0  -q

In [21]:
!pip uninstall torchvision -y
!pip install torchvision --upgrade

Found existing installation: torchvision 0.22.0
Not uninstalling torchvision at /usr/lib/python3/dist-packages, outside environment /usr
Can't uninstall 'torchvision'. No files were found to uninstall.
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 58.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 2.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 7.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 17.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 8.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 KB 95.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 4.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 24.3 MB/s eta 0:00:0000:0100:01
    

In [53]:
import pandas as pd
import re

def split_into_steps(question, full_response):
    """
    Returns a list of steps for a row:
    - Splits response by double newlines (\n\n) as recommended for Qwen2.5-Math-PRM
    """
    # Split full_response by double newlines
    steps = full_response.strip().split("\n\n")
    
    # Clean up each step (remove empty strings, strip whitespace)
    steps = [s.strip() for s in steps if s.strip()]
    
    return steps


def format_for_prm(question, steps, system_prompt="Please reason step by step, and put your final answer within \\boxed{}."):
    """
    Format question and steps for Qwen2.5-Math-PRM model.
    Returns the messages format expected by the model.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
        {"role": "assistant", "content": "<extra_0>".join(steps) + "<extra_0>"},
    ]
    return messages


# Example: read CSV
df = pd.read_csv("strategyqa_confidence_instruct.csv")

# Create new column for step-by-step response (list of steps)
df['steps'] = df.apply(lambda row: split_into_steps(row['question'], row['full_response']), axis=1)

# Create formatted messages for PRM
df['prm_messages'] = df.apply(lambda row: format_for_prm(row['question'], row['steps']), axis=1)

# Save
df.to_csv("Steps_Qwen_Instruct_StratQA.csv", index=False)

# Preview
import pprint
print("=== Steps ===")
pprint.pprint(df['steps'].iloc[0])
print("\n=== PRM Messages ===")
pprint.pprint(df['prm_messages'].iloc[0])

=== Steps ===
["Let's break this down step by step:",
 '1. **Understanding Boolean Algebra**: Boolean algebra is a branch of algebra '
 'in which the values of the variables are the truth values true and false, '
 'usually denoted 1 and 0 respectively. It deals with logical operations such '
 'as AND, OR, and NOT.',
 '2. **Binary System**: The binary system is a numeral system that represents '
 'numeric values using two symbols, typically 0 and 1. This system is '
 'fundamental to digital computing and information theory.',
 '3. **Connection Between Boolean Algebra and Binary**: In the context of '
 'digital electronics and computer science, Boolean algebra is often used to '
 'describe and manipulate binary data. The values 0 and 1 in Boolean algebra '
 'directly correspond to the binary digits (bits) in the binary system.',
 '4. **Conclusion**: While Boolean algebra can be applied to and is closely '
 'related to the binary system, it is not solely defined by it. Boolean '
 'algebra

In [54]:
!python prm_scoring.py

Loading PRM model...
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:02<00:00,  1.44it/s]
Some weights of the model checkpoint at Qwen/Qwen2.5-Math-PRM-7B were not used when initializing Qwen2ForProcessRewardModel: ['lm_head.weight']
- This IS expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Model loaded!
Computing step rewards...
100%|███████████████████████████████████████████| 50/50 [00:03<00:00, 16.53it/s]
Saved!

=== Summary ===
Mean of min step rewards: 0.6920
Mean of mean step rewards: 0.8949

Correct answers - mean step reward: 0.9055
Wrong answe